In [1]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import plotly.express as px
from flipside import Flipside
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import time
from memory_profiler import profile
import json
import csv
import os
import logging
import sys
from importlib import reload
from pymongo import MongoClient
from web3 import Web3
import psycopg2
from psycopg2.extras import execute_values
from io import StringIO
from typing import Any, Dict
from keys import KEYS

In [ ]:
# logging configurations
reload(logging)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

## Extract Data

In [4]:
# Initilize Flipside Client
flipside_key = KEYS['flipside_key']
flipside = Flipside(flipside_key, "https://api-v2.flipsidecrypto.xyz")

In [5]:
def format_query(query_path: str, params: dict) -> str:
    try:
        with open(query_path, "r") as file:
            query = file.read()
        formatted_query = query.format(**params)
        return formatted_query
    except Exception as e:
        raise ValueError(f"format query error: {e}")

In [6]:
def createQueryRun(query : str, api_key:str = flipside_key) -> str :
    
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }

    # Request payload
    payload = {
        "jsonrpc": "2.0",
        "method": "createQueryRun",
        "params": [
            {
                "resultTTLHours": 1,
                "maxAgeMinutes": 0,
                "sql": query ,
                "tags": {
                    "source": "postman-demo",
                    "env": "test"
                },
                "dataSource": "snowflake-default",
                "dataProvider": "flipside"
            }
        ],
        "id": 1
    }

    # Submit createQueryRun request
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        if response.status_code == 200:
            logging.info("Query run created successfully!")
            logging.debug(response.json())  # Output the response
            return  response.json()['result']['queryRequest']['queryRunId']
        else:
            raise requests.exceptions.HTTPError(
                f"Unexpected status code: {response.status_code}. Details: {response.text}" )
    except Exception as e:
        logging.error(f" createQueryRun Error: {e}")
    

In [7]:
def getQueryRun(queryRunId:str , api_key:str = flipside_key) -> str:
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }
    
    payload = {
    "jsonrpc": "2.0",
    "method": "getQueryRun",
    "params": [
        {
            "queryRunId": queryRunId
        }
    ],
    "id": 1
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        logging.debug(f'getQueryRun state {response.json()['result']['queryRun']['state']}')
        return response.json()['result']['queryRun']['state']
        
    except Exception as e:
        logging.error(f" getQueryRun Error: {e}")

In [8]:
def queryresult_Pagination(queryRunId:str, page_size:int = 70000) -> list:
       
    current_page_number = 1
    total_pages = 3

    all_rows = []

    while current_page_number <= total_pages:

        try:
            results = flipside.get_query_results(
                queryRunId,
                page_number=current_page_number,
                page_size=page_size         
            )
       
            if results.records:
                total_pages = results.page.totalPages
                all_rows.extend(results.records)
                logging.debug(f"Current page number: {current_page_number} Total Pages: {total_pages}, Rows Retrieved: {len(results.records)}")
            else: 
                logging.warning('No record')
                break

        except Exception as e:
            logging.error(f" Pagination Error: {e}")
            return None
        
        current_page_number += 1

    logging.info(f"Total Pages: {total_pages}, Rows Retrieved: {len(all_rows)}")
        
        
    return all_rows

In [15]:
def extract_flipsidecrypto_data(query_path:str, params: dict , api_key = flipside_key, retry_time:int = 90 ,timeout:int = 600 ) -> list:
    
    try:
        logging.info(f'Start query with params:{params}')
        query = format_query(query_path,params)
        queryRunId = createQueryRun(query,api_key)

        state = None
        start_time = time.time()

        while state != 'QUERY_STATE_SUCCESS':
            
            state = getQueryRun(queryRunId,api_key)

            if state == 'QUERY_STATE_SUCCESS':
                 break 

            elif state in ['QUERY_STATE_FAILED', 'QUERY_STATE_CANCELED']:
                raise RuntimeError(f"Query execution failed or was canceled. State: {state}")
            
            elif state in ['QUERY_STATE_STREAMING_RESULTS', 'QUERY_STATE_RUNNING', 'QUERY_STATE_READY']:
                if time.time() - start_time > timeout:
                    raise TimeoutError("Query execution exceeded timeout limit.")
                
                logging.info(f"Wainting query excution")
                logging.debug(f"retry after {retry_time} sec")

                time.sleep(retry_time)

            else: raise ValueError(f"Unexpected query state: {state}")

            
        result = queryresult_Pagination(queryRunId)

    except TimeoutError as e:
        logging.error(f"Timeout Error: {e}")
        return None
    except RuntimeError as e:
        logging.error(f"Runtime Error: {e}")
        return None
    except Exception as e:
        logging.error(f" state Error: {e}")
        return None
    
                   

    return result

In [45]:
def get_start_block_number(pool: str, file_path: str, default: int = 0) -> int:
    try:
        with open(file_path, 'r') as file:
            block_numbers = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        logging.error(f"Error reading or parsing file {file_path}: {e}")
        return default  
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
        return default  

    block_number = block_numbers.get(pool, default)
    if isinstance(block_number, int):
        return block_number
    else:
        logging.warning(f"Invalid block number for pool '{pool}': {block_number}. Returning default value: {default}")
        return default

In [46]:
def update_start_block_number(data, file_path) -> None :
    try:
        try:
            with open(file_path, 'r') as file:
                block_numbers = json.load(file)
        except (FileNotFoundError, json.JSONDecodeError):
            block_numbers = {}
        
        
        if data:
            max_block_data = max(data, key=lambda x: x['block_number'])
            last_block_number = max_block_data['block_number']
            pool = max_block_data['pool_address']
            block_numbers[pool] = last_block_number

            logging.debug(f"Block number for pool {pool} updated to {last_block_number}")
        else:
            logging.warning(f"No events found for pool {pool}, skipping.")

    except Exception as e:
        logging.error(f"Error updating block numbers: {e}")
        
    try:
        
        with open(file_path, 'w') as file:
                json.dump(block_numbers, file, indent=4)
                logging.info(f"block number config File updated")
                
    except Exception as e:
        logging.error(f"Error updating block number config File: {e}")
        
    return None

In [47]:
def fetch_positionsData(pool_address:str, query_path:str, block_number_config_file_path:str) -> list:
    try: 
        block_number = get_start_block_number(pool_address,block_number_config_file_path)
        logging.info(f"querying data for pool : {pool_address} starting from block number: {block_number}")
        position_data = extract_flipsidecrypto_data(query_path, params= {'pool_address': pool_address,'block_number':block_number} )
        logging.info(f"position data fetched successfully for pool {pool_address}, Rows Retrieved: {len(position_data)}")
        update_start_block_number(position_data,block_number_config_file_path)
    except Exception as e:
        logging.error(f"Error fetching position data for pool: {pool_address}: {e}")
    return position_data

In [48]:
pool_address = '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640'

In [49]:
positin_data_query_path = r'sql_queries\get_position_data_query.sql'
block_number_config_file_path = 'configs/block_number_config.json'
positions_extracted_data = fetch_positionsData(pool_address,positin_data_query_path,block_number_config_file_path)

2024-10-08 10:55:42 - ERROR - Error reading or parsing file configs/block_number_config.json: Expecting value: line 1 column 1 (char 0)
2024-10-08 10:55:42 - INFO - querying data for pool : 0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640 starting from block number: 0
2024-10-08 10:55:42 - INFO - Start query with params:{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640', 'block_number': 0}
2024-10-08 10:55:43 - INFO - Query run created successfully!
2024-10-08 10:55:43 - INFO - Wainting query excution
2024-10-08 10:57:14 - INFO - Wainting query excution
2024-10-08 10:58:47 - INFO - Total Pages: 1, Rows Retrieved: 157
2024-10-08 10:58:47 - INFO - position data fetched successfully for pool 0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640, Rows Retrieved: 157
2024-10-08 10:58:47 - INFO - block number config File updated


In [50]:
# positions_extracted_data -> list[dict{}] 
positions_extracted_data[1]

{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'block_hash': '0xae740b7eb80669ab57968c64688ea7d0ea8b879a98733a72ff50d2a1f1743df6',
 'block_number': 20791090,
 'tx_hash': '0x49d7dc44b5df4f11dad380b67d1ab84eeb363aae1b5c2385ef98e64acf7dfb9d',
 'tx_index': 110,
 'contract_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'event_index': 160,
 'block_timestamp': '2024-09-20T10:06:11.000Z',
 'origin_from_address': '0xf369b182ec389613802aa8cebc56c688ed9be629',
 'origin_to_address': '0xc36442b4a4522e871399cd717abdd847ab11fe88',
 'topic0': '0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acfd67e028cd568da98982c',
 'event_name': 'Burn',
 'topics': ['0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acfd67e028cd568da98982c',
  '0x000000000000000000000000c36442b4a4522e871399cd717abdd847ab11fe88',
  '0x000000000000000000000000000000000000000000000000000000000002f13e',
  '0x000000000000000000000000000000000000000000000000000000000003096c'],
 'data': '0x000000000000000000000000000000000000

In [ ]:
pool_info_query_path = r'sql_queries\get_pool_info_query.sql'
pool_info_data = extract_flipsidecrypto_data(pool_info_query_path, params= {'pool_address': pool_address} )

2024-10-08 10:59:17 - INFO - Start query with params:{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640'}
2024-10-08 10:59:18 - INFO - Query run created successfully!
2024-10-08 10:59:18 - INFO - Wainting query excution


In [ ]:
pool_info_data[0]

{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'token0': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
 'token1': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'fee': 500,
 'tickspacing': 10,
 '__row_index': 0}